# MLSys Task 3：KV Cache 与模型属性——显存与规模的权衡

对应打卡 issue：[datawhalechina/llm-algo-leetcode #133](https://github.com/datawhalechina/llm-algo-leetcode/issues/133)

理论材料：

- [KV-Cache: The Hidden Memory Consumer](https://mlsysbook.ai/mlsysim/blog/how-much-memory-llama3.html)
- [Quantization: Not a Free Lunch](https://harvard-edge.github.io/cs249r_book_dev/mlsysim/tutorials/05_quantization.html)

核心工具：`ServingModel`、`calc_kv_cache_size`、`CompressionModel`

使用说明：

- 所有实验都是**解析仿真**（Roofline / 物理公式），不是真实 GPU benchmark，Colab CPU runtime 就能跑
- 建议顺序执行；每个实验都有**预测区**，先写下猜测再运行验证（predict before you compute）
- 实验编号与 issue 一致：E1–E5 核心，O1–O4 选做。全部包含在本 notebook 中，可按打卡等级选做


In [ ]:
# Colab 每次 new runtime 需要重新安装（约 1 分钟，纯 CPU 即可，不需要 GPU）
%pip install -q "git+https://github.com/harvard-edge/cs249r_book.git@dev#subdirectory=mlsysim"


In [ ]:
import mlsysim
from mlsysim import ureg
from mlsysim.core.units import Q_
from mlsysim.solvers import ServingModel, CompressionModel
from mlsysim.physics import calc_kv_cache_size
from mlsysim.show import table, info

print("mlsysim 版本:", mlsysim.__version__)

llama8b  = mlsysim.Models.Language.Llama3_8B
llama70b = mlsysim.Models.Language.Llama3_70B
h100     = mlsysim.Hardware.Cloud.H100
solver   = ServingModel()

head_dim_8b = llama8b.hidden_dim // llama8b.heads   # 4096 / 32 = 128

print(f"模型: {llama8b.name} | layers={llama8b.layers}, heads={llama8b.heads}, "
      f"kv_heads={llama8b.kv_heads} (GQA), hidden={llama8b.hidden_dim}")
print(f"硬件: {h100.name} | 显存={h100.memory.capacity.to('GB'):.1f}, "
      f"带宽={h100.memory.bandwidth.to('GB/s'):.0f}")


## 0. 理论速览

**KV-Cache 公式**（autoregressive 推理的隐藏内存消耗者）：

```text
KV_bytes = 2 × L × H_kv × d_head × S × B × bytes_per_elem
           │   │      │       │      │   │       └ 精度字节(FP16=2)
           │   │      │       │      │   └ batch = 并发请求数
           │   │      │       │      └ 序列长度
           │   │      │       └ 每个 head 的维度
           │   │      └ KV 头数（注意 GQA：用 kv_heads，不是 heads！）
           │   └ 层数
           └ 开头的 2 = Key 和 Value 各存一份
```

> Llama3-8B 用了 GQA：32 个 attention head 但只有 **8 个 KV head**。
> 公式里误用 heads 会把 KV-Cache 高估 4 倍——最常见的踩坑点。

**量化的作用边界**：量化只减少“字节数”，不减少“FLOPs”

| 阶段 | 瓶颈类型 | 耗时公式 | INT4 效果 |
|---|---|---|---|
| Decode | memory-bound | (权重+KV 字节) / 显存带宽 | ≈ 4× 加速 |
| Prefill | compute-bound | FLOPs / 峰值算力 | ≈ 1×（除非硬件低精度算力更强） |


---
## E1 权重显存：不同精度下模型有多大？

**预测区**（运行前先填）：
- Llama3-8B FP16 ≈ ____ GB，INT8 ≈ ____ GB，INT4 ≈ ____ GB
- Llama3-70B FP16 能放进单张 80GB H100 吗？____


In [ ]:
# E1: Model.size_in_bytes(precision) —— precision 是“每参数字节数”的 Quantity
PRECISIONS = {"FP16": "2 byte", "INT8": "1 byte", "INT4": "0.5 byte"}

rows = []
for name, bpp in PRECISIONS.items():
    size = llama8b.size_in_bytes(ureg(bpp))
    rows.append([name, bpp, f"{size.to('GB'):.1f} GB"])

table(["精度", "每参数字节", "Llama3-8B 权重"], rows)

# 验证“70B FP16 放不进单张 H100”
w70_fp16 = llama70b.size_in_bytes()                      # 默认 FP16
w70_int4 = llama70b.size_in_bytes(ureg("0.5 byte"))
cap_gb = h100.memory.capacity.to("GB").magnitude

print()
print(f"Llama3-70B FP16 权重 : {w70_fp16.to('GB'):.1f} GB")
print(f"Llama3-70B INT4 权重 : {w70_int4.to('GB'):.1f} GB")
print(f"H100 显存容量        : {cap_gb:.1f} GB")
print(f"70B FP16 单卡放得下? : {w70_fp16.to('GB').magnitude <= cap_gb}")
print(f"70B INT4 单卡放得下? : {w70_int4.to('GB').magnitude <= cap_gb}")


**E1 观察要点**（对照你的预测）：

- 8B：FP16 ≈ 16 GB → INT8 减半 → INT4 再减半；权重大小 = 参数量 × 每参数字节，纯线性
- 70B FP16 ≈ 140 GB > 80 GB：**单卡物理放不下**。出路只有两条：量化到 INT4（≈35 GB），或张量并行 TP 切分到多卡
- 这解释了为什么 70B 级模型发布当天就配套了 INT4/GQA 等压缩技术


---
## E2 单请求 KV-Cache：序列长度如何吃掉显存

**预测区**：Llama3-8B 在 4K 上下文时单个请求的 KV-Cache ≈ ____ MB？128K 时 ≈ ____ GB？

提示：先手算每 token 字节数 `2 × 32 × 8 × 128 × 2`。


In [ ]:
# E2: mlsysim.physics.calc_kv_cache_size —— 注意 n_heads 传 KV 头数（GQA）
per_tok = 2 * llama8b.layers * llama8b.kv_heads * head_dim_8b * 2
print(f"每 token 每请求 KV = 2x{llama8b.layers}x{llama8b.kv_heads}x{head_dim_8b}x2 "
      f"= {per_tok} B = {per_tok/1024:.0f} KB")
print()

rows = []
for seq in [2048, 4096, 32768, 131072]:
    kv = calc_kv_cache_size(
        n_layers=llama8b.layers, n_heads=llama8b.kv_heads, head_dim=head_dim_8b,
        seq_len=seq, batch_size=1, bytes_per_elem=2,
    )
    # 交叉验证：ServingModel 内部用的正是同一个公式
    r = solver.solve(llama8b, h100, seq_len=seq, batch_size=1, precision="fp16")
    ratio = kv.to("GB").magnitude / llama8b.size_in_bytes().to("GB").magnitude
    rows.append([f"{seq//1024}K", f"{kv.to('GB'):.3f} GB",
                 f"{r.kv_cache_size.to('GB'):.3f} GB", f"{ratio:.0%}"])

table(["序列长度", "手算 calc_kv_cache_size", "ServingModel 输出", "占 FP16 权重比例"], rows)

# 最常见的错误：把 attention 头数当成 KV 头数
kv_wrong = calc_kv_cache_size(n_layers=llama8b.layers, n_heads=llama8b.heads,
                              head_dim=head_dim_8b, seq_len=4096, batch_size=1)
kv_right = calc_kv_cache_size(n_layers=llama8b.layers, n_heads=llama8b.kv_heads,
                              head_dim=head_dim_8b, seq_len=4096, batch_size=1)
print()
print(f"正确 (kv_heads=8) : {kv_right.to('MB'):.0f} MB @4K")
print(f"错误 (heads=32)   : {kv_wrong.to('MB'):.0f} MB @4K  <- 高估 {(kv_wrong/kv_right).magnitude:.0f} 倍")


**E2 观察要点**：

- KV-Cache 与序列长度**严格线性**：2K→128K 增长 64 倍，没有技巧能绕过这个量级
- 128K 时单请求 KV ≈ 16.8 GB，**已经和 8B 模型的 FP16 权重一样大**——“隐藏的显存杀手”名副其实
- GQA（8 个 KV 头）已经把缓存缩小到 1/4；没有 GQA 的老架构会更早撞墙


---
## E3 显存预算 → 最大并发请求数（H100 80GB + Llama3-8B FP16）

**预测区**：4K 上下文时一张 H100 最多能同时服务几个请求？____

预算公式：`max_concurrent = (显存容量 − 模型权重 − 预留) ÷ 单请求KV(S)`


In [ ]:
# E3: KV-Cache 公式 + 显存预算
RESERVE = Q_("2 GB")   # 激活值 / CUDA 上下文 / 框架开销的教学近似
weights = llama8b.size_in_bytes()
available = h100.memory.capacity - weights - RESERVE
print(f"可用 KV 预算 = {h100.memory.capacity.to('GB'):.1f} - {weights.to('GB'):.1f} "
      f"- {RESERVE.to('GB'):.0f} = {available.to('GB'):.1f} GB")
print()

rows = []
for seq in [2048, 4096, 32768, 131072]:
    kv1 = calc_kv_cache_size(llama8b.layers, llama8b.kv_heads, head_dim_8b,
                             seq_len=seq, batch_size=1)
    max_b = int(available.to("GB").magnitude // kv1.to("GB").magnitude)
    rows.append([f"{seq//1024}K", f"{kv1.to('GB'):.2f} GB", max_b])

table(["上下文长度", "单请求 KV", "最大并发请求数"], rows)

# 交叉验证 4K 答案：ServingModel 的 feasible = 权重+KV <= 显存（不含预留项，所以略高）
b = 1
while solver.solve(llama8b, h100, seq_len=4096, batch_size=b).feasible:
    b += 1
print()
print(f"ServingModel 逐步试探的 4K 可行上界: batch = {b-1}")


**E3 观察要点**：

- 2K 时能服务上百路并发，128K 时只剩个位数——**并发上限随上下文长度线性崩塌**
- 决定“能服务多少用户”的不是算力、不是权重，而是 **显存容量 − KV 占用**
- 所以模型卡片上的 “128K context” 不是功能而是**一张内存账单**：生产系统必须靠 PagedAttention、KV 量化、prefix 复用来省钱


---
## E4 量化对 Decode 阶段（ITL）的影响

**预测区**：INT8 / INT4 相对 FP16 的 ITL 加速比 ≈ ____ / ____


In [ ]:
# E4: ServingModel.solve(precision=...) —— Decode 用 ITL 衡量
rows, base = [], None
for prec in ["fp16", "int8", "int4"]:
    r = solver.solve(llama8b, h100, seq_len=4096, batch_size=1, precision=prec)
    itl = r.itl.to("ms").magnitude
    base = base or itl
    rows.append([prec.upper(), f"{itl:.2f} ms",
                 f"{r.model_weights_size.to('GB'):.1f} GB",
                 f"{r.kv_cache_size.to('GB'):.2f} GB",
                 f"{base/itl:.2f}x"])

table(["精度", "ITL (seq=4K, batch=1)", "权重", "KV-Cache", "加速比 vs FP16"], rows)


**E4 观察要点**：加速比几乎精确等于**字节缩减比**（INT8≈2x，INT4≈4x）。

原因：batch=1 的 decode 是教科书级 memory-bound——每生成一个 token 都要把全部权重从 HBM 重读一遍，
`ITL ≈ (权重字节 + KV 字节) / 显存带宽`。少搬字节 = 等比例提速。


---
## E5 同样的量化对 Prefill 阶段（TTFT）呢？

**预测区**：TTFT 的 INT4 加速比 ≈ ____（提示：prefill 耗时由什么决定？）


In [ ]:
# E5: 验证“量化对计算受限阶段无效”
rows, base = [], None
for prec in ["fp16", "int8", "int4"]:
    r = solver.solve(llama8b, h100, seq_len=4096, batch_size=1, precision=prec)
    ttft = r.ttft.to("ms").magnitude
    base = base or ttft
    rows.append([prec.upper(), f"{ttft:.1f} ms", f"{base/ttft:.2f}x"])

table(["精度", "TTFT (seq=4K, batch=1)", "加速比 vs FP16"], rows)


**E5 解读（本任务最重要的概念点）**：

- **INT4 ≈ 1.0x（“0 倍加速”）**：prefill 耗时 = FLOPs ÷ 有效算力。量化不改变 FLOPs，而 mlsysim 的 H100 数据表里没有 INT4 算力条目（回退 FP16 峰值 989 TFLOP/s），所以 TTFT 纹丝不动
- **INT8 可能 ≈ 2x**：H100 注册表里有 `int8: 1979 TOPS`（2× FP16 峰值）。compute-bound 的 prefill 读到这条更快的低精度路径就被加速了——这正是官方教程 “Nuance: INT8 Tensor Cores” 警告的**二阶效应**
- **结论**：量化首先省的是**字节**。能否转化为加速，取决于撞的是哪个屋顶——memory roof（decode，直接受益）还是 compute roof（prefill，默认不受益，除非硬件有更快低精度算力）。训练同理：大批量训练是 compute-bound，所以“INT4 训练 4 倍加速”是谎言


---
## O1（选做）模型规模 8B → 70B：TTFT 和 ITL 各涨多少？

**预测区**：两个指标的涨幅都 ≈ 参数量比（8.8×）吗？还是会有差别？70B FP16 还可行吗？


In [ ]:
# O1: 先预测，再验证（对应 Quantization 教程 Exercise 1）
def probe(model):
    r = solver.solve(model, h100, seq_len=4096, batch_size=1, precision="fp16")
    return r.ttft.to("ms").magnitude, r.itl.to("ms").magnitude, r.feasible

ttft8, itl8, _ = probe(llama8b)
ttft70, itl70, ok70 = probe(llama70b)
param_ratio = (llama70b.parameters / llama8b.parameters).magnitude

print(f"参数量比 70B/8B : {param_ratio:.1f}x")
print(f"{'':8s}{'8B':>10s} {'70B':>10s} {'倍数':>8s}")
print(f"{'TTFT':8s}{ttft8:>8.1f}ms {ttft70:>8.1f}ms {ttft70/ttft8:>7.1f}x")
print(f"{'ITL':8s}{itl8:>8.2f}ms {itl70:>8.2f}ms {itl70/itl8:>7.1f}x")
print()
print(f"70B FP16 单卡可行: {ok70}   (E1 已解释: 140GB > 80GB)")


**O1 解读**：

- TTFT ∝ FLOPs ∝ 参数量 → 约 8.8×；ITL ∝ 权重字节 → 也约 8.8×
- 规模放大**不会改变两阶段的瓶颈属性**（prefill 仍 compute-bound、decode 仍 memory-bound），只是把离墙的距离缩短了 8.8 倍——原本宽裕的显存预算瞬间见底（70B FP16 直接不可行）
- “模型变大”的系统后果是双重的：每一步更慢 + 能服务的并发更少（70B 有 80 层，每 token KV 是 8B 的 2.5 倍）


---
## O2（选做）INT4 加速比存在临界 batch size 吗？

issue 预期：batch 变大后 decode 从 memory-bound 滑向 compute-bound，INT4 加速比会跌破 2x / 1.5x。

**预测区**：你认为临界 batch ≈ ____


In [ ]:
# O2: 扫 batch size 1->128，对比 FP16 vs INT4 的 ITL（对应 Quantization 教程 Exercise 2）
batches = [1, 2, 4, 8, 16, 32, 64, 96, 128]
rows, sp_hist = [], []
for b in batches:
    r16 = solver.solve(llama8b, h100, seq_len=4096, batch_size=b, precision="fp16")
    if not r16.feasible:
        rows.append([b, "OOM", "-", "-", "-"])
        continue
    r4 = solver.solve(llama8b, h100, seq_len=4096, batch_size=b, precision="int4")
    itl16 = r16.itl.to("ms").magnitude
    itl4  = r4.itl.to("ms").magnitude
    sp = itl16 / itl4
    sp_hist.append((b, sp))
    rows.append([b, f"{itl16:.2f} ms", f"{itl4:.2f} ms", f"{sp:.2f}x",
                 f"{r16.total_memory_required.to('GB'):.0f} / {cap_gb:.0f} GB"])

table(["Batch", "ITL FP16", "ITL INT4", "加速比", "FP16 显存占用/容量"], rows)

below = lambda th: next((bb for bb, s in sp_hist if s < th), None)
print()
print(f"加速比首次 < 2x 的 batch : {below(2)}")
print(f"加速比首次 < 1.5x 的 batch: {below(1.5)}")


In [ ]:
import matplotlib.pyplot as plt

if sp_hist:
    xs, ys = zip(*sp_hist)
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.plot(xs, ys, marker="o", color="#2563eb")
    ax.axhline(4.0, ls="--", c="gray", lw=1)
    ax.axhline(2.0, ls=":", c="red", lw=1)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("batch size (log2)")
    ax.set_ylabel("INT4 ITL speedup (x)")
    ax.set_title("INT4 vs FP16 decode speedup under memory budget")
    ax.grid(alpha=.3)
    plt.tight_layout()
    plt.show()


**O2 如实解读（一阶模型的边界，也是很好的学习素材）**：

- 在 decode 一阶公式 `ITL = (W + KV) / BW + 框架税` 里，W 和 KV **都按精度同比缩放**，所以加速比理论上恒等于字节比 ≈ 4x，只会被固定的框架税（32 层 × 0.01ms）略微拉低
- 实际发生的是：FP16 先撞 **OOM 边界**（看表中显存占用列，4K 上下文大约在 batch≈130 之后不可行），而 INT4 权重和 KV 都缩到 1/4，可行域大得多
- 真实系统中“加速比随 batch 跌破 2x”来自三个此模型未建模的因素：① 大 batch 后 decode 进入 compute-bound，受 Tensor Core 峰值限制；② TP>1 时通信不随精度缩减；③ 调度/采样开销固定。想看瓶颈迁移，用下面的 Engine Roofline 视角补充观察


In [ ]:
# 补充视角：Engine.solve 的 Roofline 判断（观察大 batch 下瓶颈从 Memory -> Compute）
for b in [1, 16, 64]:
    p16 = mlsysim.Engine.solve(llama8b, h100, batch_size=b, precision="fp16")
    p4  = mlsysim.Engine.solve(llama8b, h100, batch_size=b, precision="int4")
    sp = p16.latency.to("ms").magnitude / p4.latency.to("ms").magnitude
    print(f"batch={b:>3} | FP16: {p16.bottleneck:<8} {p16.latency.to('ms'):8.2f} | "
          f"INT4: {p4.bottleneck:<8} {p4.latency.to('ms'):8.2f} | speedup={sp:.2f}x")


---
## O3（选做）CompressionModel：量化 vs 剪枝的压缩比-精度权衡

注意：`CompressionModel` 以 **FP32 (4 字节)** 为基线计算压缩比，所以 INT8 显示 4x、INT4 显示 8x。


In [ ]:
# O3: 量化(INT8/INT4) vs 剪枝(sparsity=0.5/0.75/0.9)（对应 Quantization 教程 Exercise 3）
comp = CompressionModel()
rows = []

for bits in [8, 4]:
    c = comp.solve(llama8b, h100, method="quantization", target_bitwidth=bits)
    rows.append([f"量化 INT{bits}", f"{c.compression_ratio:.0f}x",
                 f"{c.compressed_size_gb.to('GB'):.1f} GB",
                 f"{c.estimated_accuracy_delta:+.2%}",
                 f"{c.inference_speedup:.1f}x"])

for sp, stype in [(0.5, "unstructured"), (0.75, "unstructured"), (0.9, "unstructured"),
                  (0.5, "structured"), (0.75, "structured"), (0.9, "structured")]:
    c = comp.solve(llama8b, h100, method="pruning", sparsity=sp, sparsity_type=stype)
    rows.append([f"剪枝 {sp:.0%} ({stype})", f"{c.compression_ratio:.1f}x",
                 f"{c.compressed_size_gb.to('GB'):.1f} GB",
                 f"{c.estimated_accuracy_delta:+.2%}",
                 f"{c.inference_speedup:.1f}x"])

table(["方案", "压缩比(vs FP32)", "压缩后大小", "估计精度变化", "推理加速"], rows)


**O3 解读框架**（结合表格数值作答）：

- **INT8 是性价比之王**：<1% 精度损失，换 2× 内存 + decode 2× 加速，几乎无条件值得
- **INT4**：8× 压缩但精度税 2–5%，适合对延迟/显存极度敏感、能接受质量回退的场景（或配合 QAT/LoRA 回收精度）
- **非结构化剪枝**：只有存储收益，`inference_speedup=1.0`——稀疏权重在没有专用硬件/内核时跑不出加速
- **结构化剪枝**：稀疏度越高加速越大，但经验规律是超过 ~50% 后精度断崖（对比 75%/90% 的 accuracy delta），风险显著高于量化
- 结论句式建议：“在我的约束（显存上限 / 精度下限）下，____ 是最优先手段，因为 ______”


---
## O4（选做）70B 部署可行性报告素材（8×H100 节点）

思路：TP=8 把权重切到每卡；GQA 的 8 个 KV head 正好每卡分摊 1 个；剩下的显存全部用来买并发。


In [ ]:
# O4: FP16 vs INT4 在 8-GPU H100 节点上的并发容量（TP=8）
reserve_gb = 2.0

w70_fp16_gb = llama70b.size_in_bytes().to("GB").magnitude
w70_int4_gb = llama70b.size_in_bytes(ureg("0.5 byte")).to("GB").magnitude

# TP=8: kv_heads=8 -> 每卡 1 个 KV head；FP16 KV 每元素 2B
kv70_tok_card_fp16_gb = 2 * llama70b.layers * (llama70b.kv_heads // 8) * 128 * 2 / 1e9
kv70_tok_card_int4_gb = kv70_tok_card_fp16_gb / 4   # 权重与 KV 均按 4-bit 计

rows = []
for prec, w_pc, kv_tok in [("FP16", w70_fp16_gb / 8, kv70_tok_card_fp16_gb),
                           ("INT4", w70_int4_gb / 8, kv70_tok_card_int4_gb)]:
    for seq in [2048, 4096, 16384]:
        kv1 = kv_tok * seq
        free = cap_gb - w_pc - reserve_gb
        rows.append([prec, f"{seq//1024}K", f"{w_pc:.1f}", f"{kv1:.3f}",
                     int(free // kv1)])

table(["精度", "上下文", "每卡权重 GB", "每请求 KV GB (TP=8)", "每卡最大并发"], rows)
print()
print("节点总并发 = 每卡并发 x 8 卡")


**O4 报告模板**（把上表数字填进去就是一份合格的部署建议）：

1. **用什么精度？** 建议 ____ 起步：每卡权重仅 ____ GB，给 KV 留出 ____ GB 预算；若精度敏感可退回 FP16 + 更短上下文
2. **最大支持多长上下文？** 16K 时 FP16 每卡并发跌至 ____，INT4 仍有 ____ → 生产上以 ____ 为宜
3. **最多服务多少并发？** 4K 目标下 8 卡节点合计约 ____ 路；要再往上需 PagedAttention（消除碎片 +20–40%）或 KV INT8（再翻倍）
4. **风险提示**：以上为解析仿真的一阶估算，未含激活值峰值、TP 通信对 ITL 的拖累与碎片化；上线前需真实流量压测


---
## 打卡对照清单（issue #133）

**最小打卡（E1–E3）**
- [ ] E1：三种精度权重大小表 + “70B FP16 放不进单张 H100”的一句话证明
- [ ] E2：四档序列长度 KV 表 + 线性增长说明
- [ ] E3：各长度最大并发表 + “显存容量是核心约束”一段话

**学有余力 1（+E4 E5 O1 O2）**
- [ ] E4/E5：两张精度对比表 + 解释“为什么 Decode 4x 而 Prefill 0x”（记得提 INT8 二阶效应）
- [ ] O1：8B→70B 的 TTFT/ITL 倍数
- [ ] O2：加速比-batch 曲线 + 你找到的临界点/OOM 边界

**学有余力 2（+O3 O4）**
- [ ] O3：量化 vs 剪枝对比表 + 性价比结论
- [ ] O4：三问部署建议（精度/上下文/并发）

> 截图建议：保留每个 `table(...)` 输出与关键 `print` 结果，GitHub 打卡评论按 E1→O4 顺序贴图。
